# CHƯƠNG 5. XÂY DỰNG VÀ HUẤN LUYỆN MÔ HÌNH PHÂN LỚP

## 5.1. Xây dựng các mô hình phân lớp

### Khai báo thư viện

In [70]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

### Đọc dữ liệu

In [54]:
plt.rcParams["figure.figsize"] = (8, 6)

RANDOM_STATE = 42
TEST_SIZE = 0.2
VALID_SIZE = 0.2

BASE_DIR = Path("..")
DATA_PATH = BASE_DIR / "pima-indians-diabetes.csv"
OUT_DIR = BASE_DIR / "outputs"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
MODEL_DIR = OUT_DIR / "models"

TARGET_COL = "Outcome"
ALL_COLS = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome",
]
ZERO_AS_MISSING_COLS = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

for p in [OUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, header=None, names=ALL_COLS)
    return df

In [46]:
df = load_data(DATA_PATH)
print("Đã đọc dữ liệu xong")
print("Kích thước:", df.shape)
print(df.head())
print("\nPhân bố nhãn:")
print(df["Outcome"].value_counts())

Đã đọc dữ liệu xong
Kích thước: (768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Phân bố nhãn:
Outcome
0    500
1    268
Name: count, dtype: int64


## 5.1 Xây dựng mô hình phân lớp

In [47]:
NUMERIC_FEATURES = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
]

def get_models_and_grids() -> Dict[str, Tuple[object, Dict[str, List[object]]]]:
    return {
        "LogisticRegression": (
            LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
            {
                "model__C": [0.01, 0.1, 1.0, 10.0],
                "model__solver": ["liblinear"],
                "model__penalty": ["l2"],
            },
        ),
        "KNN": (
            KNeighborsClassifier(),
            {
                "model__n_neighbors": [3, 5, 7, 9, 11, 15],
                "model__weights": ["uniform", "distance"],
                "model__p": [1, 2],
            },
        ),
        "DecisionTree": (
            DecisionTreeClassifier(random_state=RANDOM_STATE),
            {
                "model__max_depth": [None, 3, 5, 7, 10],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4],
                "model__criterion": ["gini", "entropy"],
            },
        ),
        "RandomForest": (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
            {
                "model__n_estimators": [100, 200, 300],
                "model__max_depth": [None, 5, 10],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4],
            },
        ),
    }

In [ ]:
models = get_models_and_grids()
print("Danh sách mô hình:")
for name in models:
    print("-", name)

Danh sách mô hình:
- LogisticRegression
- KNN
- DecisionTree
- RandomForest


## 5.2 Huấn luyện mô hình

### tiền xử lý đầu vào

In [55]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ZERO_AS_MISSING_COLS:
        df.loc[df[col] == 0, col] = np.nan
    return df

@dataclass
class SplitData:
    X_train: pd.DataFrame
    X_valid: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_valid: pd.Series
    y_test: pd.Series

### Chia dữ liệu train / validation / test

In [50]:
def split_data(df: pd.DataFrame) -> SplitData:
    X = df.drop(columns=[TARGET_COL])
    y = df[TARGET_COL].astype(int)

    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    valid_ratio_from_trainval = VALID_SIZE / (1.0 - TEST_SIZE)
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_trainval, y_trainval,
        test_size=valid_ratio_from_trainval,
        random_state=RANDOM_STATE,
        stratify=y_trainval,
    )

    return SplitData(X_train, X_valid, X_test, y_train, y_valid, y_test)

In [53]:
df_clean = clean_data(df)
split = split_data(df_clean)
print("Đã chia dữ liệu")
print("Train:", split.X_train.shape, split.y_train.shape)
print("Valid:", split.X_valid.shape, split.y_valid.shape)
print("Test :", split.X_test.shape, split.y_test.shape)
print("\nTỷ lệ nhãn train:")
print(split.y_train.value_counts(normalize=True))

Đã chia dữ liệu
Train: (460, 8) (460,)
Valid: (154, 8) (154,)
Test : (154, 8) (154,)

Tỷ lệ nhãn train:
Outcome
0    0.652174
1    0.347826
Name: proportion, dtype: float64


In [56]:
def build_preprocessor(scaler_name: str | None) -> ColumnTransformer:
    if scaler_name == "minmax":
        scaler = MinMaxScaler()
    elif scaler_name == "standard":
        scaler = StandardScaler()
    elif scaler_name is None:
        scaler = "passthrough"
    else:
        raise ValueError(f"scaler_name không hợp lệ: {scaler_name}")

    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", scaler),
                    ]
                ),
                NUMERIC_FEATURES,
            )
        ],
        remainder="drop",
    )


### Huấn luyện trên dữ liệu thô / Min-Max / Standard

In [63]:
df_clean = clean_data(load_data(DATA_PATH))
split = split_data(df_clean)

versions = {
    "raw": None,
    "minmax": "minmax",
    "standard": "standard",
}

results = []
best_models = {}

for version_name, scaler_name in versions.items():
    print("\n" + "="*80)
    print(f"ĐANG HUẤN LUYỆN PHIÊN BẢN: {version_name.upper()}")

    preprocessor = build_preprocessor(scaler_name)

    for model_name, (model, grid) in get_models_and_grids().items():
        print("-"*80)
        print(f"Model: {model_name}")

        pipe = Pipeline([
            ("preprocess", preprocessor),
            ("model", model),
        ])

        search = GridSearchCV(
            estimator=pipe,
            param_grid=grid,
            cv=5,
            scoring="accuracy",
            n_jobs=-1,
        )
        search.fit(split.X_train, split.y_train)

        valid_pred = search.best_estimator_.predict(split.X_valid)
        valid_acc = accuracy_score(split.y_valid, valid_pred)

        print(f"Best CV Accuracy: {search.best_score_:.4f}")
        print(f"Valid Accuracy   : {valid_acc:.4f}")
        print(f"Best Params      : {search.best_params_}")

        results.append({
            "version": version_name,
            "model": model_name,
            "cv_best_acc": search.best_score_,
            "valid_acc": valid_acc,
            "best_params": search.best_params_,
        })

        best_models[(version_name, model_name)] = search.best_estimator_
        joblib.dump(search.best_estimator_, MODEL_DIR / f"{version_name}_{model_name}.joblib")


ĐANG HUẤN LUYỆN PHIÊN BẢN: RAW
--------------------------------------------------------------------------------
Model: LogisticRegression


c:\Users\PhatTB\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best CV Accuracy: 0.7783
Valid Accuracy   : 0.7792
Best Params      : {'model__C': 10.0, 'model__penalty': 'l2', 'model__solver': 'liblinear'}
--------------------------------------------------------------------------------
Model: KNN
Best CV Accuracy: 0.7652
Valid Accuracy   : 0.7532
Best Params      : {'model__n_neighbors': 15, 'model__p': 1, 'model__weights': 'distance'}
--------------------------------------------------------------------------------
Model: DecisionTree
Best CV Accuracy: 0.7435
Valid Accuracy   : 0.7597
Best Params      : {'model__criterion': 'entropy', 'model__max_depth': 3, 'model__min_samples_leaf': 4, 'model__min_samples_split': 2}
--------------------------------------------------------------------------------
Model: RandomForest
Best CV Accuracy: 0.7761
Valid Accuracy   : 0.7857
Best Params      : {'model__max_depth': None, 'model__min_samples_leaf': 4, 'model__min_samples_split': 2, 'model__n_estimators': 100}

ĐANG HUẤN LUYỆN PHIÊN BẢN: MINMAX
--------------

c:\Users\PhatTB\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best CV Accuracy: 0.7630
Valid Accuracy   : 0.7857
Best Params      : {'model__n_neighbors': 15, 'model__p': 2, 'model__weights': 'uniform'}
--------------------------------------------------------------------------------
Model: DecisionTree
Best CV Accuracy: 0.7435
Valid Accuracy   : 0.7597
Best Params      : {'model__criterion': 'entropy', 'model__max_depth': 3, 'model__min_samples_leaf': 4, 'model__min_samples_split': 2}
--------------------------------------------------------------------------------
Model: RandomForest
Best CV Accuracy: 0.7739
Valid Accuracy   : 0.7857
Best Params      : {'model__max_depth': None, 'model__min_samples_leaf': 4, 'model__min_samples_split': 2, 'model__n_estimators': 100}

ĐANG HUẤN LUYỆN PHIÊN BẢN: STANDARD
--------------------------------------------------------------------------------
Model: LogisticRegression
Best CV Accuracy: 0.7891
Valid Accuracy   : 0.7922
Best Params      : {'model__C': 0.1, 'model__penalty': 'l2', 'model__solver': 'liblinear'}

c:\Users\PhatTB\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best CV Accuracy: 0.7609
Valid Accuracy   : 0.7922
Best Params      : {'model__n_neighbors': 11, 'model__p': 2, 'model__weights': 'uniform'}
--------------------------------------------------------------------------------
Model: DecisionTree
Best CV Accuracy: 0.7435
Valid Accuracy   : 0.7597
Best Params      : {'model__criterion': 'entropy', 'model__max_depth': 3, 'model__min_samples_leaf': 4, 'model__min_samples_split': 2}
--------------------------------------------------------------------------------
Model: RandomForest
Best CV Accuracy: 0.7739
Valid Accuracy   : 0.7857
Best Params      : {'model__max_depth': 5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 300}


### Kết quả huấn luyện

In [72]:
results_df = pd.DataFrame(results).sort_values(
    ["version", "valid_acc"],
    ascending=[True, False],
)

# 1. Thông tin tổng quan
print("\n" + "="*80)
print("BẢNG KẾT QUẢ HUẤN LUYỆN (VALIDATION)")
print("Sắp xếp theo: version ↑ và valid_acc ↓")

display(results_df)

# 2. Model tốt nhất của mỗi version
print("\n" + "="*80)
print("MODEL TỐT NHẤT THEO TỪNG PHIÊN BẢN DỮ LIỆU:")

best_by_version = results_df.groupby("version").first()

for version, row in best_by_version.iterrows():
    print(f"\n[{version.upper()}]")
    print(f"Model      : {row['model']}")
    print(f"Valid Acc  : {row['valid_acc']:.4f}")
    print(f"CV Score   : {row['cv_best_acc']:.4f}")
    print(f"Params     : {row['best_params']}")

# 3. Model tốt nhất toàn bộ
best_overall = results_df.iloc[0]

print("\n" + "="*80)
print("MODEL TỐT NHẤT TOÀN BỘ:")
print(f"Version    : {best_overall['version']}")
print(f"Model      : {best_overall['model']}")
print(f"Valid Acc  : {best_overall['valid_acc']:.4f}")
print(f"CV Score   : {best_overall['cv_best_acc']:.4f}")
print(f"Params     : {best_overall['best_params']}")

# 4. Lưu file
results_df.to_excel(TABLE_DIR / "validation_results.xlsx", index=False)
print("\nĐã lưu bảng kết quả tại:", TABLE_DIR / "validation_results.xlsx")


BẢNG KẾT QUẢ HUẤN LUYỆN (VALIDATION)
Sắp xếp theo: version ↑ và valid_acc ↓


,version,model,cv_best_acc,valid_acc,best_params
4,minmax,LogisticRegression,0.786957,0.792208,"{'model__C': 10.0, 'model__penalty': 'l2', 'mo..."
5,minmax,KNN,0.763043,0.785714,"{'model__n_neighbors': 15, 'model__p': 2, 'mod..."
7,minmax,RandomForest,0.773913,0.785714,"{'model__max_depth': None, 'model__min_samples..."
6,minmax,DecisionTree,0.743478,0.759740,"{'model__criterion': 'entropy', 'model__max_de..."
3,raw,RandomForest,0.776087,0.785714,"{'model__max_depth': None, 'model__min_samples..."
0,raw,LogisticRegression,0.778261,0.779221,"{'model__C': 10.0, 'model__penalty': 'l2', 'mo..."
2,raw,DecisionTree,0.743478,0.759740,"{'model__criterion': 'entropy', 'model__max_de..."
1,raw,KNN,0.765217,0.753247,"{'model__n_neighbors': 15, 'model__p': 1, 'mod..."
8,standard,LogisticRegression,0.789130,0.792208,"{'model__C': 0.1, 'model__penalty': 'l2', 'mod..."
9,standard,KNN,0.760870,0.792208,"{'model__n_neighbors': 11, 'model__p': 2, 'mod..."



MODEL TỐT NHẤT THEO TỪNG PHIÊN BẢN DỮ LIỆU:

[MINMAX]
Model      : LogisticRegression
Valid Acc  : 0.7922
CV Score   : 0.7870
Params     : {'model__C': 10.0, 'model__penalty': 'l2', 'model__solver': 'liblinear'}

[RAW]
Model      : RandomForest
Valid Acc  : 0.7857
CV Score   : 0.7761
Params     : {'model__max_depth': None, 'model__min_samples_leaf': 4, 'model__min_samples_split': 2, 'model__n_estimators': 100}

[STANDARD]
Model      : LogisticRegression
Valid Acc  : 0.7922
CV Score   : 0.7891
Params     : {'model__C': 0.1, 'model__penalty': 'l2', 'model__solver': 'liblinear'}

MODEL TỐT NHẤT TOÀN BỘ:
Version    : minmax
Model      : LogisticRegression
Valid Acc  : 0.7922
CV Score   : 0.7870
Params     : {'model__C': 10.0, 'model__penalty': 'l2', 'model__solver': 'liblinear'}

Đã lưu bảng kết quả tại: ..\outputs\tables\validation_results.xlsx


# Kết thúc